In [ ]:
%load_ext autoreload
%autoreload 2
# activate line execution
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"



In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import numpy as np
import pickle


In [ ]:
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
# path to the mimic-iii-clinical-database-1.4
# this is only used for selecting ICD codes
MIMIC_PATH = "/mlodata1/hokarami/tedam/mimic3/mimic-iii-clinical-database-1.4/"



## ICD codes
* we extract the ICD codes (Diagnosis and Procedure codes) based on a minimum frequency threshold

In [ ]:
MIN_TH_ICD = 5 # minimum threshold for an ICD code to be considered

In [ ]:
df_admissions = pd.read_csv(os.path.join(MIMIC_PATH, "ADMISSIONS.csv")).sort_values(['SUBJECT_ID', 'ADMITTIME'])
df_patients = pd.read_csv(os.path.join(MIMIC_PATH, "PATIENTS.csv")).sort_values(['SUBJECT_ID'])


In [ ]:
# for DIAGNOSES_ICD

df_icd = pd.read_csv(os.path.join(MIMIC_PATH, "DIAGNOSES_ICD.csv")).sort_values(['SUBJECT_ID', 'HADM_ID', 'SEQ_NUM'])
df_icd.columns

df_icd['ICD9_CODE'].value_counts().head(10)

freqs = df_icd['ICD9_CODE'].value_counts()#/len(df_icd)*100
freqs = freqs[freqs>MIN_TH_ICD]
freqs.shape

# plot the distribution of the frequency of icd codes
fig = px.bar(freqs.head(10),  title='Top 10 ICD9 codes (Dianosis)')
fig.show()

df_icd = df_icd[df_icd['ICD9_CODE'].isin(freqs.index)]



In [ ]:
# for PROCEDURES_ICD

df_proc = pd.read_csv(os.path.join(MIMIC_PATH, "PROCEDURES_ICD.csv")).sort_values(['SUBJECT_ID', 'HADM_ID', 'SEQ_NUM'])
df_proc.columns

# change dtype from int to object
df_proc['ICD9_CODE'] = df_proc['ICD9_CODE'].astype(str)

df_proc['ICD9_CODE'].value_counts().head(10)

freqs = df_proc['ICD9_CODE'].value_counts()#/len(df_proc)*100
freqs = freqs[freqs>MIN_TH_ICD]
freqs.shape

# plot the distribution of the frequency of icd codes
fig = px.bar(freqs.head(10),  title='Top 10 ICD9 codes (Procedures)')
fig.show()

df_proc = df_proc[df_proc['ICD9_CODE'].isin(freqs.index)]

# MIMIC-III Benchmarks

* we use processed data from mimic3 benchmark (https://github.com/YerevaNN/mimic3-benchmarks)
* we consider two tasks: in-hospital mortality prediction (IHM) and phenotyping (PHE)


[n_patients * {sid, label_ihm, label_phe, covariates, events, ts}]

In [ ]:
# specify the paths for the benchmark datasets

PATH_ROOT = "/mlodata1/hokarami/mimic3-benchmarks/data/root4"
PATH_IHM = "/mlodata1/hokarami/mimic3-benchmarks/data/in-hospital-mortality2"
PATH_PHE = "/mlodata1/hokarami/mimic3-benchmarks/data/phenotyping2"

# specify the paths for the processed data
PATH_DATA = "processed/mimic3"


## Variables


In [ ]:

############################################ Time series variables ############################################

# categorical time series variables
TS_CAT = ['Capillary refill rate','Glascow coma scale eye opening','Glascow coma scale motor response', 'Glascow coma scale total',
       'Glascow coma scale verbal response']

# continuous time series variables
TS_CONT = [ 'Alanine aminotransferase', 'Albumin', 'Alkaline phosphate', 'Anion gap', 'Asparate aminotransferase', 'Basophils', 'Bicarbonate', 'Bilirubin', 'Blood urea nitrogen', 'Calcium', 'Chloride', 'Creatinine', 'Diastolic blood pressure', 'Fraction inspired oxygen', 'Glucose', 'Heart Rate', 'Hematocrit', 'Hemoglobin', 'Lactate', 'Lymphocytes', 'Mean blood pressure', 'Mean corpuscular volume', 'Monocytes', 'Neutrophils', 'Oxygen saturation', 'Partial pressure of carbon dioxide', 'Phosphate', 'Platelets', 'Potassium', 'Prothrombin time', 'Red blood cell count', 'Respiratory rate', 'Systolic blood pressure', 'Temperature', 'White blood cell count', 'pH']

# timestamp variable

TS_VARS = TS_CAT+TS_CONT
TS_VARS_types = ['categorical']*len(TS_CAT) + ['continuous']*len(TS_CONT)
TS_HOURS = ['Hours']
print(f"Number of categorical time series variables: {len(TS_CAT)}")
print(f"Number of continuous time series variables: {len(TS_CONT)}")




############################################ Covariates ############################################
# categorical covariates
COVARS_CAT = ['Gender']

# continuous covariates
COVARS_CONT = ['Age']

COVARS = COVARS_CONT + COVARS_CAT
COVARS_types = ['continuous']*len(COVARS_CONT) + ['categorical']*len(COVARS_CAT)


print(f"Number of categorical covariates: {len(COVARS_CAT)}")
print(f"Number of continuous covariates: {len(COVARS_CONT)}")

In [ ]:


df_all = pd.read_csv(PATH_ROOT + "/all_stays.csv")
print('root', df_all.shape[0])

for task in ['in-hospital-mortality2', 'phenotyping2']:
    n = 0
    for split in ['train', 'test','val']:
        path = f"/mlodata1/hokarami/mimic3-benchmarks/data/{task}/{split}_listfile.csv"
        # path = f"/mlodata1/hokarami/mimic3-benchmarks/data/{task}/{split}_listfile.csv"
        df = pd.read_csv(path)
        # df.shape
        n+= df.shape[0]
    print(task, n)



In [ ]:
phe_train = pd.read_csv(PATH_PHE + "/train_listfile.csv")
ihm_train = pd.read_csv(PATH_IHM + "/train_listfile.csv")

phe_train.shape
ihm_train.shape


df = phe_train.merge(ihm_train, on='stay', how='outer')

df.shape


# so ihm is a subset of phe

In [ ]:
# Re-structuring the data from mimic3-benchmarks to a more suitable format for the task

for split in ['train','test','val']:
    df_split = pd.read_csv(PATH_PHE + f"/{split}_listfile.csv")
    
    data = []
    list_sids = []
    for _, row in tqdm(df_split.iloc[:].iterrows(), total=df_split.shape[0]):
        stay = row.stay
        sid = int(stay.split('_')[0])
        order = int(stay.split('_')[1][7:])-1 # addmision id, 0-indexed
        label_phe = row[2:].values.astype(int)
        
        

        # getting hadm_id and label_ihm from root directory
        ff = 'test' if split=='test' else 'train'
        stay_csv = pd.read_csv(PATH_ROOT+f"/{ff}/{sid}/stays.csv")
        
        hadm_id = stay_csv.iloc[order].HADM_ID        
        label_ihm = stay_csv.iloc[order].MORTALITY_INHOSPITAL        
        
        
        
        covariates = [
            stay_csv.iloc[order].AGE,
            1 if stay_csv.iloc[order].GENDER=='M' else 0,
        ]
        
        # icd and proc codes
        codes_icd = ['icd_' + x for x in df_icd[df_icd.HADM_ID == hadm_id]['ICD9_CODE'].tolist()]
        codes_proc = ['proc_'+ x for x in df_proc[df_proc.HADM_ID == hadm_id]['ICD9_CODE'].tolist()]
        codes = [codes_icd, codes_proc]

        # ts data
        df_ts = pd.read_csv(PATH_PHE+f"/{ff}/{stay}")[TS_VARS+TS_HOURS]



        
        if sid in list_sids: # if the subject is already in the list
            
            subject_data = data[list_sids.index(sid)]

            subject_data['hadm_id'].append(hadm_id)
            subject_data['covariates'].append(covariates)
            subject_data['codes'].append(codes)
            subject_data['ts'].append(df_ts)
            subject_data['label_ihm'].append(label_ihm)
            subject_data['label_phe'].append(label_phe)
            
            
        else: # if the subject is not in the list
            subject_data = {
                'sid': sid, # subject id
                'hadm_id': [hadm_id], # admission id
                'covariates': [covariates], # covariates
                'codes': [codes], # icd and proc codes
                'ts': [df_ts], # time series data
                'label_ihm': [label_ihm], # ihm label
                'label_phe': [label_phe]    # phe label

            }
            list_sids.append(sid)

            data.append(subject_data)
    

    
    # save to pickle
    print(f"len data {split}:", len(data))
    with open(f"{PATH_DATA}/{split}Dataset.pkl", "wb") as f:
        pickle.dump(data, f)
    
    

In [ ]:
# an example

subject_data

# Token Dictionary

* Before tokenization, we need to create a dictionary of tokens

In [ ]:

token2id = {} # maps token to id
var2id = {} # for non-codes only

## Events

ICD codes (diagnosis and procedures) 

In [ ]:
freqs_icd = df_icd['ICD9_CODE'].value_counts()
freqs_icd.index = 'icd_' + freqs_icd.index



freqs_proc = df_proc['ICD9_CODE'].value_counts()
freqs_proc.index = 'proc_' + freqs_proc.index



freqs = pd.concat([freqs_icd, freqs_proc])
freqs

codeToId = {code: i for i, code in enumerate(freqs.index)}
idToCode = {i: code for i, code in enumerate(freqs.index)}

# list_tokens.extend(freqs.index.tolist())
# vocab_size['codes'] = len(freqs)
# len(list_tokens)

assert len(token2id) == 0, "token2id should be empty"
token2id.update({('code',i): i for i in range(len(freqs))})

token2id


## time series

In [ ]:
BIN_TYPE = 'uniform'  # 'uniform' or 'quantile'


In [ ]:
# reading time series data from train split

data = pickle.load(open(f"{PATH_DATA}/trainDataset.pkl", "rb"))
temp = pd.concat([ts for patient in data for ts in patient['ts']])


In [ ]:
# addressing some issues in the data

# convert all categorical variables to string
for var in TS_CAT:
    temp[var] = temp[var].astype(str)

# replace 'nan' with np.nan
temp[temp=='nan'] = np.nan


# convert all continuous variables to numeric
for col in tqdm(TS_CONT+TS_HOURS):
    temp[col] = pd.to_numeric(temp[col], errors='coerce')
    
temp.dtypes



In [ ]:
# this is a fix for some categorical variables

def find_possible_values(all_values, var):

    vals = all_values.unique().tolist()

    dict_map = None
    tokens = []

    if var=='Glascow coma scale eye opening':
        dict_map = {'1 No Response': 0,
            '3 To speech': 1,
            '4 Spontaneously': 2,
            '2 To pain': 3,
            'To Speech': 1,
            'Spontaneously': 2,
            'To Pain': 3}
        tokens = ['No Response', 'To Speech', 'Spontaneously', 'To Pain']
        tokens = ["GCS-eo-"+x for x in tokens]

    elif var=='Glascow coma scale motor response':
        dict_map = {'5 Localizes Pain': 0,
            '6 Obeys Commands': 1,
            '4 Flex-withdraws': 2,
            '1 No Response': 3,
            '2 Abnorm extensn': 4,
            '3 Abnorm flexion': 5,
            'Abnormal extension': 4,
            'Obeys Commands': 1,
            'Localizes Pain': 0,
            'No response': 3,
            'Flex-withdraws': 2,
            'Abnormal Flexion': 5}
        tokens = ['Localizes Pain', 'Obeys Commands', 'Flex-withdraws', 'No Response', 'Abnormal extension', 'Abnormal Flexion']
        tokens = ["GCS-mr-"+x for x in tokens]

    elif var=='Glascow coma scale verbal response':

        dict_map = {'1.0 ET/Trach': 0,
            '5 Oriented': 1,
            '4 Confused': 2,
            '1 No Response': 3,
            '2 Incomp sounds': 4,
            '3 Inapprop words': 5,
            'No Response-ETT': 0,
            'Oriented': 1,
            'Inappropriate Words': 5,
            'Confused': 2,
            'Incomprehensible sounds': 4,
            'No Response': 3}
        tokens = ['ET/Trach', 'Oriented', 'Confused', 'No Response', 'Incomp sounds', 'Inapprop words']
        tokens = ["GCS-vr-"+x for x in tokens]
    else:
        dict_map = {val: i for i, val in enumerate(vals)}
        tokens = [f"{var}-{x}" for x in vals]

    return dict_map, tokens

In [ ]:
# This  section is used for soft labels (experimental)

from scipy.ndimage import convolve1d

def soft_label_with_gaussian(i, N, kernel_size, sigma=1.0):
    """
    Generate a soft label vector with Gaussian smoothing and softmax normalization.
    
    Args:
    i (int): Index for one-hot encoding.
    N (int): Length of the one-hot encoded vector.
    kernel_size (int): Size of the Gaussian kernel (should be odd).
    sigma (float): Standard deviation of the Gaussian kernel (default is 1.0).
    
    Returns:
    soft_label (np.array): Soft label vector of length N.
    """
    # Step 1: One-Hot Encoding
    one_hot = np.zeros(N)
    one_hot[i] = 1
    # print(one_hot)

    # Step 2: Create Gaussian Kernel
    n_neighbour = (kernel_size - 1) // 2
    x = np.linspace(-n_neighbour, n_neighbour, kernel_size)
    gaussian_kernel = np.exp(-0.5 * (x / sigma) ** 2)
    gaussian_kernel /= gaussian_kernel.sum()  # Normalize the kernel

    # print(gaussian_kernel)
    # Step 3: Convolution with Gaussian Kernel
    soft_label = convolve1d(one_hot, gaussian_kernel, mode='constant')

    # # Step 4: Apply Softmax
    # soft_label = np.exp(soft_label)  # Exponentiate to ensure non-negative values
    # soft_label /= soft_label.sum()   # Normalize to make sum = 1

    # Apply sum normalization
    soft_label = soft_label / soft_label.sum()
    return soft_label

# Example usage:
i = 3
N = 5
kernel_size = 7
soft_label = soft_label_with_gaussian(i, N, kernel_size, sigma=1.0)
print("Soft Label:", soft_label)

for i in range(5):
    soft_label_with_gaussian(i, N, kernel_size, sigma=1.0)


#### TS tokenization

In [ ]:
assert  len(token2id)==len(codeToId), "Error: please run 'Token Dictionary' again"


ts_info = {}

beginPos = [0]
possibleValues = {}
variableRanges={}
discretization = {}
isCategorical={}

var2id = {} # only variables that have multiple values

soft_labels = {}
for i, var in tqdm(enumerate( TS_CAT    + TS_CONT), total=len(TS_CAT    + TS_CONT)):
    # print(var)


    all_values = temp[var]
    missing_rate = pd.isnull(all_values).sum()/len(all_values)
    all_values = all_values[all_values.notnull()] # remove nans
    
    n_unique  = all_values.nunique()

    if var in TS_CAT:
        var_type = 'categorical'
    else:
        var_type = 'continuous'
    ts_info[var] = {
        # 'n_unique': n_unique,
        'var_type': var_type,
        'missing_rate': missing_rate
    }


    if var_type == 'categorical':
        isCategorical.update({var: True})
        
        
        dict_map, tokens = find_possible_values(all_values, var)
        
        # list_tokens.extend(tokens)
        n_tokens = len(tokens)

        var2id.update({var: len(var2id)})
        token2id.update({('ts',var2id[var],i): i+len(token2id) for i in range(n_tokens)})


        beginPos.append(beginPos[-1] + n_tokens) 

        possibleValues.update({var:dict_map})

        fig = go.Figure()
        _ = fig.add_trace(go.Histogram(x=all_values, histnorm='probability', name=f"{var}-{var_type}", opacity=0.75))

        fig.write_html(PATH_DATA + f"/ts/_{var}.html")

        ts_info[var].update({
            'unique_values' : tokens,
            'n_tokens': n_tokens
        })
    else:
        isCategorical.update({var: False})
        
        # remove str elements
        # all_values2 = all_values[all_values.apply(lambda x: not isinstance(x, str))]
        all_values2 = all_values
        # print(f'str elements % for {var}: {(len(all_values) - len(all_values2))/len(all_values)*100}')

        ts_info[var]['mean'] = all_values2.mean()
        ts_info[var]['std'] = all_values2.std()
        ts_info[var]['min'] = all_values2.min()
        ts_info[var]['max'] = all_values2.max()
        ts_info[var]['0.025'] = all_values2.quantile(0.025)
        ts_info[var]['0.975'] = all_values2.quantile(0.975)
        # median
        ts_info[var]['median'] = all_values2.median()

        

        if (ts_info[var]['max'] - ts_info[var]['0.975'])/(ts_info[var]['0.975'] - ts_info[var]['median']) > 3:
            all_values2 = all_values2[all_values2<=ts_info[var]['0.975']]
        if (ts_info[var]['0.025'] - ts_info[var]['min'])/(ts_info[var]['median'] - ts_info[var]['0.025']) > 3:
            all_values2 = all_values2[all_values2>=ts_info[var]['0.025']]
        

        ts_info[var]['mean_2'] = all_values2.mean()
        ts_info[var]['std_2'] = all_values2.std()
        ts_info[var]['min_2'] = all_values2.min()
        ts_info[var]['max_2'] = all_values2.max()
        ts_info[var]['0.025_2'] = all_values2.quantile(0.025)
        ts_info[var]['0.975_2'] = all_values2.quantile(0.975)
        # median
        ts_info[var]['median_2'] = all_values2.median()
        
        n_bins = min(10, int(n_unique/5))

        if BIN_TYPE == 'uniform':
            binned_data, bin_edges = pd.cut(all_values2, bins=n_bins, retbins=True, duplicates='drop')
        elif BIN_TYPE == 'quantile':
            binned_data, bin_edges = pd.qcut(all_values2,n_bins, retbins=True, duplicates='drop')
        
        
        beginPos.append(beginPos[-1] + n_bins)
        
        
        
        discretization.update({var: bin_edges.tolist()})
        possibleValues.update({var: {f"{var}_{i}":i for i in range(n_bins) }})
        var2id.update({var: len(var2id)})
        token2id.update({('ts',var2id[var],i): i+len(token2id) for i in range(n_bins)})



        # creat soft labels
        kernel_size = 7
        for i in range(n_bins):
                    soft_labels.update({('ts',var2id[var],i):     soft_label_with_gaussian(i, n_bins, kernel_size, sigma=1.0)
        })


        # bin_edges


        bin_counts, _ = np.histogram(all_values2, bins=bin_edges)
        total_count = len(all_values2)
        normalized_bin_counts = bin_counts / total_count

        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2





        # plot distribution using plotly
        fig = go.Figure()

        _ = fig.add_trace(go.Bar(x=bin_centers, y=normalized_bin_counts,width=np.diff(bin_edges)*1, name=f"{var}-{var_type}", opacity=0.5))

        _ = fig.add_trace(go.Histogram(x=all_values2, histnorm='probability', name=f"{var}-{var_type}", opacity=0.75))

        # fig.show()
        
        # save fig to PATH_DATA
        fig.write_html(PATH_DATA + f"/ts/{var}.html")


        ts_info[var].update(
            {
                'n_tokens': n_bins,
                'bin_edges': bin_edges.tolist(),
                'bin_counts': bin_counts.tolist(),

                'normalized_bin_counts': normalized_bin_counts.tolist(),
                'bin_centers': bin_centers.tolist(),
                'bin_labels': [f"{var}_{i}" for i in range(n_bins)]
            }
        )



# remove last elelemt from beginPos
beginPos.pop()

In [ ]:
ts_info

In [ ]:
len(token2id)
token2id.keys()
token2id[('ts',var2id['Heart Rate'],0)]



In [ ]:
# see a summary of time series variables that are tokenized

info = pd.DataFrame(ts_info).T

info.columns

ttt = [ 'var_type', 'missing_rate', 'min','min_2','0.025','median', '0.975','max','max_2','n_tokens']
info[ttt].head(10)

### Covars & Labels

In [ ]:

# We manually tokenize some other variables


possibleValues.update({
    "Gender": {0: 0, 1: 1},
})
isCategorical.update({
    "Age": False,
    "Gender": True,
    "Hours": False,
})
discretization.update({
    "Age": [18, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90],
    "Days": [0, 11, 16, 21, 25, 30.1, 35.1, 43, 48, 54, 60, 66, 72, 81, 90, 100.1],
    "Hours": [
        0,
        0.5,
        1.5,
        2.5,
        3.5,
        6.5,
        10.5,
        16.5,
        26.5,
        48.0,
        48.1,
        60.1,
        80.1,
        110.1,
        150.1,
        200.1,
    ],
})



# add age
var2id.update({"Age": len(var2id)})
token2id.update({('covar',var2id["Age"],i): i+len(token2id) for i in range(len(discretization["Age"])-1)})

# add gender
var2id.update({"Gender": len(var2id)})
token2id.update({('covar',var2id["Gender"],i): i+len(token2id) for i in range(2)})

# add hours
var2id.update({"Hours": len(var2id)})
token2id.update({('timestamp',var2id["Hours"],i): i+len(token2id) for i in range(len(discretization["Hours"])-1)})


# add label tokens
token2id.update({('label','phe',i): len(token2id)+i for i in range(25)})
token2id.update({('label','ihm',i): len(token2id)+i for i in range(2)})


### Special tokens

In [ ]:

# add start record token
token2id.update({'<s>': len(token2id)}) # start record token

# add end covar token
token2id.update({'</covar>': len(token2id)}) # end covar token

# add end labels
token2id.update({'</label>': len(token2id)}) # end label token

# add end codes
token2id.update({'</code>': len(token2id)}) # end code token

# add end ts token
token2id.update({'</ts>': len(token2id)}) # end ts token

# add end adm token
token2id.update({'</adm>': len(token2id)}) # end adm token

token2id.update({'</s>': len(token2id)}) # end record token
# add end record token

# add pad token
token2id.update({'<pad>': len(token2id)}) # pad token



################################# THESE ARE EXPERIMENTAL
# add start history token
token2id.update({'<history>': len(token2id)}) # add start history token 

# add start forecast token
token2id.update({'<forecast>': len(token2id)}) # add start forecast token

# add end forecast token
token2id.update({'</forecast>': len(token2id)}) # add end forecast token




In [ ]:
token2id['<s>']

In [ ]:
# EXPERIMENTAL
# create soft_label matrix

len(soft_labels)

M_soft_labels = np.eye(len(token2id))
for token in token2id.keys():
    if token in soft_labels:
        temp = soft_labels[token]
        max_pos = np.argmax(temp)
        l_left = token2id[token] - max_pos
        l_right = token2id[token] + len(temp) - max_pos
        
        M_soft_labels[token2id[token], l_left:l_right] = temp
M_soft_labels.shape

In [ ]:
# extract the name of phenotypes

phe_names = pd.read_csv(PATH_PHE + f"/train/listfile.csv").columns[2:].tolist()
idToLabel = {i: label for i, label in enumerate(phe_names)}

In [ ]:
vocab_size = {
    'codes': len(codeToId),
    'lab_cont': sum([ts_info[var]['n_tokens'] for var in TS_CAT]),
    'lab_cat': sum([ts_info[var]['n_tokens'] for var in TS_CONT]),
    'gap': len(discretization['Hours'])-1,
    'covars': len(discretization['Age'])-1 + 2,
}

vocab_size

In [ ]:
# save
metadata = {'codeToId': codeToId, 'idToCode': idToCode, 'ts_info': ts_info,
            # 'idToLab': idToLab, 'labToNumber': labToNumber, 
            "token2id": token2id, 'var2id': var2id, 
            
            'beginPos': beginPos, 'possibleValues': possibleValues, 'isCategorical': isCategorical, 'discretization': discretization,
            'idToLabel': idToLabel,
            'vocab_size': vocab_size,
            'M_soft_labels': M_soft_labels
            }

if BIN_TYPE == 'uniform':
    with open(PATH_DATA + "/metadata2.pkl", "wb") as f:
        pickle.dump(metadata, f)

else:
    with open(PATH_DATA + "/metadata.pkl", "wb") as f:
        pickle.dump(metadata, f)

# raw time series
We extract the full dataframe time series of each split and save it separately. This will be used in the RESULTS.ipynb notebook to compute the performance of the models.

In [ ]:
ts_info = pickle.load(open(PATH_DATA + "/metadata2.pkl", "rb"))['ts_info']


for split in ['train','test','val']:
    with open(f"{PATH_DATA}/{split}Dataset.pkl", "rb") as f:
        data = pickle.load(f)
    df_concat = pd.concat([ts for patient in data for ts in patient['ts']])
    print(f" data {split} loaded")

    list_concat = []

    # only first admission is considered
    for ii, patient in tqdm(enumerate(data[:])): 
        temp = patient['ts'][0]
        temp['id'] = ii
        temp['Age'] = patient['covariates'][0][0]
        temp['Gender'] = patient['covariates'][0][1]
        temp['label_ihm'] = patient['label_ihm'][0]
        
        phe_labels = [f'label_phe_{i}' for i in range(len(patient['label_phe'][0]))]
        temp[phe_labels] = patient['label_phe'][0]


        list_concat.append(temp)
    df_concat = pd.concat(list_concat)


    # addressing some issues in the data

    # convert all categorical variables to string
    for var in TS_CAT:
        df_concat[var] = df_concat[var].astype(str)

    # replace 'nan' with np.nan
    df_concat[df_concat=='nan'] = np.nan


    # convert all continuous variables to numeric
    for col in tqdm(TS_CONT+TS_HOURS):
        df_concat[col] = pd.to_numeric(df_concat[col], errors='coerce')

    for var in TS_CONT:
        min2 = ts_info[var]['min_2'] # corresponds to 2.5% percentile
        max2 = ts_info[var]['max_2']
        
        # now limit the column to min2, max2
        df_concat[var] = df_concat[var].clip(min2, max2)

    # save to csv
    df_concat.to_csv(f"{PATH_DATA}/{split}TS.csv", index=False)
    print(f" data {split} saved in csv")
    

# Discretize

In [ ]:
# choose the bin type

BIN_TYPE = 'uniform' # 'uniform' or 'quantile'

In [ ]:
PATH_DATA

# load json file
if BIN_TYPE == 'uniform':
    print("loading metadata2")
    with open(PATH_DATA + "/metadata2.pkl", "rb") as f:
        metadata = pickle.load(f)
else:
    with open(PATH_DATA + "/metadata.pkl", "rb") as f:
        metadata = pickle.load(f)



possibleValues = metadata['possibleValues']
discretization = metadata['discretization']
var2id = metadata['var2id']
token2id = metadata['token2id']
isCategorical = metadata['isCategorical']
codeToId = metadata['codeToId']
ts_info = metadata['ts_info']

id2token = {v: k for k, v in token2id.items()}

id2var = {v: k for k, v in var2id.items()}

In [ ]:
# some helper functions

def get_index(mapping, key, value):
    # this function returns the index of the value in the mapping[key]
    possible_values = mapping[key]
    for i in range(len(possible_values) - 1):
        if value <= possible_values[i + 1]:
            return i
    if value > possible_values[-1]:
        return len(possible_values) - 2
    print(f"{value} for {key} not in {possible_values}")
    return int(len(possible_values)-2)

def add_ts_data():
    # this function discretizes the time series data
    global bad_data

    adm_ts = []

    prev_time = 0
    # FLAG_24 = False
    # horizon = 1
    for time, mes in df_ts.iterrows():
        mes = {k:v for k,v in mes.items() if not pd.isnull(v)}
        # print(time)
        # print(mes)
        new_labs = []
        new_values = []
        for var, val in mes.items():
            if isCategorical[var]:
                new_labs.append(
                    var2id[var]
                )  # var2id[var] is the index in [17]
                try:
                    new_values.append(
                        possibleValues[var][str(val)]
                    )  # why always the negative value? added +1 for correction
                except:
                    print(f"Error Categorical: {var} {val}")
            else: # continuous

                try:
                    new_values.append(get_index(discretization, var, float(val)))
                    new_labs.append(var2id[var])
                except:
                    bad_data+=1
                    print(f"Error Cont: {var} {val}")
        
        

        time_gap = get_index(discretization, "Hours", time-prev_time)

            
        prev_time = time
        if len(new_labs) == len(new_values) and len(new_labs)>0:
            adm_ts.append((new_labs, new_values, [time_gap]))  # v[0] is empty
        else:
            # print("Error: different length of new_labs and new_values")
            pass
    return adm_ts


In [ ]:
# TOKENIZE THE DATA

bad_data = 0


HORIZONS = [12,24,36,48] # EXPERIMENTAL



for split in [ 'train', 'val','test']:
    data = pickle.load(open(f"{PATH_DATA}/{split}Dataset.pkl", "rb"))
    
    disc_data = []
    for p in tqdm(data[:]):
        all_covars = []
        all_codes = []
        all_ts = []
        all_horizons = []
        for i_stay in range(len(p['hadm_id'])):
            new_code = []
            covariates = p['covariates'][i_stay]
            
            # def add_covars():
                
            x = []
            y = []
            for var in COVARS:
                x.append(
                    var2id[var]
                )
                if isCategorical[var]:
                    
                    
                    y.append(
                            possibleValues[var][(covariates[COVARS.index(var)])]
                        ) 
                else:
                    y.append(get_index(discretization, var,  covariates[COVARS.index(var)]))
                    # x.append(get_index(discretization, var, covariates[COVARS.index(var)]))

            all_covars.append((
                
                x,
                y,
                

            ))
            
            
            

            codes = p['codes'][i_stay]
            
            new_code = [codeToId[code] for code in (codes[0] + codes[1])]
            # code_ids

            # new_code.append(code_ids)
            all_codes.append(new_code)



            df_ts = p['ts'][i_stay].set_index('Hours')
            

            hours = df_ts.index.tolist()


            list_horizons = []
            for horizon in HORIZONS:
                # find the maximum hour that is smaller than horizon
                try:
                    max_hour = max([x for x in hours if x<horizon])
                    max_hour_id = hours.index(max_hour)
                except:
                    print(f"Error: {horizon} {hours}")
                    max_hour_id = -1
                # max_hour,    hours.index(max_hour)
                list_horizons.append(max_hour_id)
            all_horizons.append(list_horizons)
            
            
            adm_ts = add_ts_data()

            if horizon == 1:
                a=1
            
            all_ts.append(adm_ts)


            
        # new_visits
        disc_patient = {
            'covars': all_covars,
            'codes': all_codes,
            'ts': all_ts,
            'labels_phe': p['label_phe'],
            'labels_ihm': p['label_ihm'],
            'horizons': all_horizons
        }
        disc_data.append(disc_patient)

    # save to pickle

    
    if BIN_TYPE == 'uniform':
        print("saving uniform")
        with open(f"{PATH_DATA}/{split}DiscDataset.pkl", "wb") as f:
            pickle.dump(disc_data, f)
    else:
        with open(f"{PATH_DATA}/{split}DiscDatasetQuant.pkl", "wb") as f:
            pickle.dump(disc_data, f)

    print(f"bad data: {bad_data}")
        

In [ ]:
# an example
disc_data[0]